# DeepCritical Research Agent Demo

This notebook demonstrates the DeepCritical research agent using a Gradio interface.
Backend: [DeepCritical](https://github.com/DeepCritical/DeepCritical)

In [ ]:
# @title 1. Setup Environment
# @markdown Clone the repository and install dependencies. This may take a few minutes.

!git clone -b dev https://github.com/DeepCritical/DeepCritical.git
%cd DeepCritical
!pip install .
!pip install gradio nest_asyncio

In [ ]:
# @title 2. Import Libraries
import os
import sys
import gradio as gr
import hydra
from omegaconf import DictConfig, OmegaConf
import asyncio
import nest_asyncio

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

# Ensure the current directory is in python path
sys.path.append(os.getcwd())

try:
    from DeepResearch.app import run_graph
except ImportError:
    print("Could not import run_graph. Ensure you are in the DeepCritical directory.")


In [ ]:
# @title 3. Define Research Function

def run_research(
    question,
    openai_key,
    anthropic_key,
    tavily_key,
    flow_selection
):
    """
    Runs the deep research workflow based on user input.
    """
    # Set environment variables
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
    if anthropic_key:
        os.environ["ANTHROPIC_API_KEY"] = anthropic_key
    if tavily_key:
        os.environ["TAVILY_API_KEY"] = tavily_key
    
    try:
        # Clear any existing hydra instance
        try:
            hydra.core.global_hydra.GlobalHydra.instance().clear()
        except:
            pass
        
        # Initialize Hydra with the configs directory
        # We use relative path 'configs' assuming we are in the repo root
        with hydra.initialize(version_base=None, config_path="configs"):
            # Base config overrides
            overrides = [
                f"question={question}",
            ]
            
            # Enable specific flows based on selection
            if flow_selection == "PRIME (Protein Engineering)":
                overrides.append("flows.prime.enabled=true")
            elif flow_selection == "Bioinformatics":
                overrides.append("flows.bioinformatics.enabled=true")
            elif flow_selection == "DeepSearch":
                overrides.append("flows.deepsearch.enabled=true")
            elif flow_selection == "Challenge":
                overrides.append("challenge.enabled=true")
            elif flow_selection == "RAG":
                overrides.append("flows.rag.enabled=true")
            
            # Compose config
            cfg = hydra.compose(config_name="config", overrides=overrides)
            
            # Run the graph
            # Note: run_graph is async but wrapped in asyncio.run inside DeepResearch/app.py usually,
            # but let's check the implementation we saw earlier.
            # run_graph in app.py does: loop.run_until_complete(g.run(...))
            # If we are already in an event loop (like in Jupyter), this might be tricky.
            # However, app.py creates a new event loop: loop = asyncio.new_event_loop()
            # This usually works fine unless nest_asyncio is needed.
            
            result = run_graph(question, cfg)
            return result

    except Exception as e:
        import traceback
        return f"Error occurred: {str(e)}\n\nTraceback:\n{traceback.format_exc()}"


In [ ]:
# @title 4. Launch Gradio Interface

with gr.Blocks(title="DeepCritical Research Agent") as demo:
    gr.Markdown("# DeepCritical Research Agent")
    gr.Markdown("An autonomous research agent powered by Hydra and Pydantic Graph.")
    
    with gr.Row():
        with gr.Column():
            openai_key = gr.Textbox(label="OpenAI API Key", type="password", placeholder="sk-...")
            anthropic_key = gr.Textbox(label="Anthropic API Key", type="password", placeholder="sk-ant-...")
            tavily_key = gr.Textbox(label="Tavily API Key", type="password", placeholder="tvly-...")
    
    with gr.Row():
        question = gr.Textbox(label="Research Question", placeholder="e.g. What are the core contributions of the PRIME paper?", lines=3)
    
    with gr.Row():
        flow_selection = gr.Dropdown(
            choices=["Default", "PRIME (Protein Engineering)", "Bioinformatics", "DeepSearch", "Challenge", "RAG"],
            value="Default",
            label="Research Flow"
        )
    
    submit_btn = gr.Button("Start Research", variant="primary")
    
    output = gr.Markdown(label="Research Report")
    
    submit_btn.click(
        fn=run_research,
        inputs=[question, openai_key, anthropic_key, tavily_key, flow_selection],
        outputs=output
    )

demo.launch(debug=True, share=True)
